# DQ 检查 — ODS / DWD 层（Rules 1-12）

Rules 1-10: ODS 完整性/范围/时序
Rules 11-12: DWD 行数守恒 + 参照完整性

任何 FAIL 将抛出异常，阻断 DWS 下游任务。

## 1. SparkSession

In [ ]:
import os
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("TLC-DQ-AfterDWD")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

BASE          = os.path.expanduser("~/tlc-spark-pipeline")
ODS_Y         = f"{BASE}/data/warehouse/ods/yellow"
ODS_G         = f"{BASE}/data/warehouse/ods/green"
ODS_F         = f"{BASE}/data/warehouse/ods/fhv_hv"
DWD_FACT      = f"{BASE}/data/warehouse/dwd/fact_trips"
DWD_ZONE      = f"{BASE}/data/warehouse/dwd/dim_zone"
DWD_CLEANING  = f"{BASE}/data/warehouse/dq/dwd_cleaning"   # records rejected by DWD cleaning
QUARANTINE    = f"{BASE}/data/warehouse/dq/quarantine"

TARGET_DATE = os.environ.get("TARGET_DATE", "2024-03-15")
yr  = int(TARGET_DATE.split("-")[0])
mo  = int(TARGET_DATE.split("-")[1])
day = int(TARGET_DATE.split("-")[2])
print(f"TARGET_DATE={TARGET_DATE}")

print(f"Spark {spark.version} ready | TARGET_DATE={TARGET_DATE}")


## 2. DQ 框架

In [ ]:
results = []

def check(rule_id, layer, rule_type, description, passed, bad_count=0, total=0, threshold=None, severity="block"):
    status = "PASS" if passed else "FAIL"
    results.append({
        "rule_id": rule_id, "layer": layer, "rule_type": rule_type,
        "description": description, "status": status,
        "bad_count": bad_count, "total": total, "threshold": threshold,
        "severity": severity,
    })
    mark  = "✅" if passed else ("⚠️ " if severity == "warn" else "❌")
    label = " [WARN]" if (not passed and severity == "warn") else ""
    print(f"{mark} Rule {rule_id:02d} [{layer}]{label} {description}  bad={bad_count:,}")

## 3. Rules 1-10：ODS 层检查（Yellow + Green）
FHV HV 字段名不同（base_passenger_fare / trip_miles / PUlocationID），单独在第 5 节检查。

In [3]:
ods_y = spark.read.parquet(ODS_Y).filter(
    (F.col("pickup_year") == yr) & (F.col("pickup_month") == mo) & (F.col("pickup_day") == day))
ods_g = spark.read.parquet(ODS_G).filter(
    (F.col("pickup_year") == yr) & (F.col("pickup_month") == mo) & (F.col("pickup_day") == day))
ods   = ods_y.unionByName(ods_g, allowMissingColumns=True)
ods_total = ods.count()
print(f"ODS (yellow+green) total: {ods_total:,}")

# FHV HV 单独读取，字段名不同，后续独立检查
ods_f = spark.read.parquet(ODS_F).filter(
    (F.col("pickup_year") == yr) & (F.col("pickup_month") == mo) & (F.col("pickup_day") == day))
ods_f_total = ods_f.count()
print(f"ODS (fhv_hv) total:      {ods_f_total:,}")

# Rules 1-4: 关键字段严格不允许 null
for rule_id, field in enumerate(
    ["fare_amount", "pickup_datetime", "PULocationID", "DOLocationID"],
    start=1
):
    bad = ods.filter(col(field).isNull()).count()
    check(rule_id, "ODS", "completeness", f"{field} not null",
          passed=(bad == 0), bad_count=bad, total=ods_total)

# Rule 5: passenger_count null 率告警（< 15%）
# null 记录车费/时间/区域均完整，属真实行程 → 打标，不隔离
bad5 = ods.filter(col("passenger_count").isNull()).count()
check(5, "ODS", "completeness_warn",
      "passenger_count null rate < 15% (tag-only, not quarantined)",
      passed=(bad5 / ods_total < 0.15), bad_count=bad5, total=ods_total, threshold=0.15)

# Rule 6: fare_amount > 0
bad6 = ods.filter(col("fare_amount") <= 0).count()
check(6, "ODS", "range", "fare_amount > 0", passed=(bad6 / ods_total < 0.05),
      bad_count=bad6, total=ods_total, threshold=0.05)

# Rule 7: trip_distance >= 0
bad7 = ods.filter(col("trip_distance") < 0).count()
check(7, "ODS", "range", "trip_distance >= 0", passed=(bad7 == 0),
      bad_count=bad7, total=ods_total)

# Rule 8: passenger_count in [1,6]（null 单独由 Rule 5 处理，此处只检查非 null 越界）
bad8 = ods.filter(
    col("passenger_count").isNotNull() & ~col("passenger_count").between(1, 6)
).count()
check(8, "ODS", "range", "passenger_count in [1,6]", passed=(bad8 / ods_total < 0.15),
      bad_count=bad8, total=ods_total, threshold=0.15)

# Rule 9: dropoff > pickup
bad9 = ods.filter(col("dropoff_datetime") <= col("pickup_datetime")).count()
check(9, "ODS", "temporal", "dropoff > pickup", passed=(bad9 / ods_total < 0.01),
      bad_count=bad9, total=ods_total)

# Rule 10: 时间在 2024 Q1
bad10 = ods.filter(
    (col("pickup_datetime") < "2024-01-01") | (col("pickup_datetime") >= "2024-04-01")
).count()
check(10, "ODS", "temporal", "pickup in 2024-Q1", passed=(bad10 / ods_total < 0.01),
      bad_count=bad10, total=ods_total)


ODS (yellow+green) total: 130,849


ODS (fhv_hv) total:      740,231


✅ Rule 01 [ODS] fare_amount not null  bad=0


✅ Rule 02 [ODS] pickup_datetime not null  bad=0


✅ Rule 03 [ODS] PULocationID not null  bad=0


...

✅ Rule 06 [ODS] fare_amount > 0  bad=2,246


✅ Rule 07 [ODS] trip_distance >= 0  bad=0


✅ Rule 08 [ODS] passenger_count in [1,6]  bad=1,516


✅ Rule 09 [ODS] dropoff > pickup  bad=40


✅ Rule 10 [ODS] pickup in 2024-Q1  bad=0


In [ ]:
# ── Rules 13-16：ODS 层检查（FHV HV 专属字段）─────────────────────────────
fhv_cols = set(ods_f.columns)
print(f"FHV HV columns: {sorted(fhv_cols)}\n")

if "trip_time" in fhv_cols:
    bad13 = ods_f.filter(col("trip_time") <= 0).count()
    check(13, "ODS_FHV", "range", "trip_time > 0",
          passed=(bad13 / ods_f_total < 0.01), bad_count=bad13, total=ods_f_total)
else:
    print("⚠️  trip_time not found — Rule 13 skipped")

if "trip_miles" in fhv_cols:
    bad14 = ods_f.filter(col("trip_miles") <= 0).count()
    check(14, "ODS_FHV", "range", "trip_miles > 0",
          passed=(bad14 / ods_f_total < 0.01), bad_count=bad14, total=ods_f_total)
else:
    print("⚠️  trip_miles not found — Rule 14 skipped")

if "driver_pay" in fhv_cols:
    bad15 = ods_f.filter(col("driver_pay") < 0).count()
    check(15, "ODS_FHV", "range", "driver_pay >= 0",
          passed=(bad15 / ods_f_total < 0.01), bad_count=bad15, total=ods_f_total)
else:
    print("⚠️  driver_pay not found — Rule 15 skipped")

if "request_datetime" in fhv_cols:
    bad16 = ods_f.filter(
        col("request_datetime").isNotNull() &
        (col("request_datetime") > col("pickup_datetime"))
    ).count()
    # warn-only: ~1% clock drift is systematic upstream noise — records tagged in fact, not quarantined
    check(16, "ODS_FHV", "temporal",
          "request_datetime <= pickup_datetime (tag-only in fact, not quarantined)",
          passed=(bad16 / ods_f_total < 0.03), bad_count=bad16, total=ods_f_total,
          threshold=0.03, severity="warn")
else:
    print("⚠️  request_datetime not found — Rule 16 skipped")

## 4. Rules 11-12：DWD 层检查

In [ ]:
fact       = spark.read.parquet(DWD_FACT).filter(
    (F.col("pickup_year") == yr) & (F.col("pickup_month") == mo) & (F.col("pickup_day") == day))
fact_count = fact.count()
dim_zone   = spark.read.parquet(DWD_ZONE)
print(f"fact_trips: {fact_count:,}")

# Rule 11: True conservation — fact + dwd_cleaning == ods_all_total (no silent drops)
# dwd_cleaning holds records rejected during DWD cleaning (written by 03_dwd_build)
# If this check fails, records were silently dropped — a bug, not a quality filter
try:
    dwd_cleaning = spark.read.parquet(DWD_CLEANING).filter(
        (F.col("pickup_year") == yr) & (F.col("pickup_month") == mo) & (F.col("pickup_day") == day))
    rejected_count = dwd_cleaning.count()
except Exception:
    rejected_count = 0
    print("⚠️  DWD_CLEANING not found — run 03_dwd_build first to populate it")

ods_all_total  = ods_total + ods_f_total
accounted_rows = fact_count + rejected_count
gap            = ods_all_total - accounted_rows

conservation_ok = (gap == 0)
check(11, "DWD", "row_conservation",
      f"fact ({fact_count:,}) + dwd_cleaning ({rejected_count:,}) == ods_all ({ods_all_total:,})",
      passed=conservation_ok,
      bad_count=abs(gap), total=ods_all_total)

if not conservation_ok:
    print(f"  ⚠️  {abs(gap):,} records {'silently dropped' if gap > 0 else 'extra'} — "
          "check 03_dwd_build for untracked filter paths")

# Rule 12: 参照完整性 PULocationID ∈ dim_zone
valid_zones = set(r.location_id for r in dim_zone.collect())
bad12 = fact.filter(~col("PULocationID").isin(list(valid_zones))).count()
check(12, "DWD", "referential", "PULocationID in dim_zone",
      passed=(bad12 == 0), bad_count=bad12, total=fact_count)


## 5. 违规记录写入隔离表

In [ ]:
# 读取 DWD_CLEANING（03_dwd_build 写出的隔离表）作为唯一的 DQ 记录来源
# 不在此处重新计算规则 —— 两套独立规则会导致统计数字不一致
try:
    dwd_cl = spark.read.parquet(DWD_CLEANING).filter(
        (F.col("pickup_year") == yr) & (F.col("pickup_month") == mo) & (F.col("pickup_day") == day))
    q_total = dwd_cl.count()
except Exception as e:
    print(f"⚠️  DWD_CLEANING unreadable: {e}")
    print("   Run 03_dwd_build first.")
    q_total = 0
    dwd_cl  = None

ods_all = ods_total + ods_f_total
q_rate  = q_total / ods_all if ods_all > 0 else 0

print(f"ODS total (yellow+green): {ods_total:,}")
print(f"ODS total (fhv_hv):       {ods_f_total:,}")
print(f"DWD_CLEANING rejected:    {q_total:,}")
print(f"Overall rejection rate:   {q_rate:.2%}")

if dwd_cl is not None and q_total > 0:
    print("\nRejection breakdown by source + reason:")
    dwd_cl.groupBy("source_type", "dq_reason").count() \
          .orderBy("source_type", F.desc("count")).show(30)

    for src, ods_cnt in [("yellow", ods_y.count()), ("green", ods_g.count()), ("fhv_hv", ods_f_total)]:
        src_q = dwd_cl.filter(F.col("source_type") == src).count()
        print(f"  {src}: {src_q:,} / {ods_cnt:,} = {src_q/ods_cnt:.2%}")


## 6. DQ 报告 + 阻断判断

In [ ]:
# Persist DQ results before spark.stop() and before any raise
DQ_RESULTS_PATH = os.path.expanduser("~/tlc-spark-pipeline/data/warehouse/dq/dq_results")
if results:
    from pyspark.sql import Row
    rows = [Row(
        check_date=TARGET_DATE,
        rule_id=str(r["rule_id"]),
        layer=r.get("layer", ""),
        rule_type=r.get("rule_type", ""),
        description=r["description"],
        status=r["status"],
        severity=r.get("severity", "block"),
        bad_count=int(r.get("bad_count", 0) or 0),
        total_count=int(r.get("total", 0) or 0),
        threshold=float(r["threshold"]) if r.get("threshold") is not None else None,
        gate="gate2_dwd",
        pickup_year=yr,
        pickup_month=mo,
        pickup_day=day,
    ) for r in results]
    df = spark.createDataFrame(rows)
    df = df.withColumn("bad_rate",
        F.when(F.col("total_count") > 0,
               F.round(F.col("bad_count") / F.col("total_count"), 6))
         .otherwise(F.lit(None)))
    spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
    (df.write
       .partitionBy("gate", "pickup_year", "pickup_month", "pickup_day")
       .mode("overwrite")
       .parquet(DQ_RESULTS_PATH))
    print(f"DQ results persisted: {len(rows)} rules → {DQ_RESULTS_PATH}")

In [ ]:
block_fails = [r for r in results if r['status'] == 'FAIL' and r.get('severity', 'block') == 'block']
warn_fails  = [r for r in results if r['status'] == 'FAIL' and r.get('severity', 'block') == 'warn']
pass_rules  = [r for r in results if r['status'] == 'PASS']

print(f"\n{'='*50}")
print(f"DQ Report (ODS+DWD)  {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"Rules: {len(results)} total | {len(pass_rules)} PASS | {len(block_fails)} BLOCK-FAIL | {len(warn_fails)} WARN")
print(f"DWD_CLEANING rejection rate: {q_rate:.2%}  ({q_total:,} / {ods_all:,})")
print(f"={'='*49}")

spark.stop()

if warn_fails:
    print("Non-blocking warnings:")
    for r in warn_fails:
        print(f"  ⚠️  Rule {r['rule_id']:02d}: {r['description']}  bad={r['bad_count']:,}")

if block_fails:
    print("Blocking failures:")
    for r in block_fails:
        print(f"  ❌ Rule {r['rule_id']:02d}: {r['description']}  bad={r['bad_count']:,}")
    raise ValueError(
        f"DQ FAILED: {len(block_fails)} blocking rule(s). "
        "Pipeline halted — fix upstream data before proceeding to DWS."
    )
else:
    print("No blocking failures — proceeding to DWS.")

print("DQ after-DWD check complete.")